# GitHub Repository RAG

### Retrieval-Augmented Generation (RAG) system for querying and interacting with the contents of GitHub repositories.

---

**Author:** David Fernández Martínez  
**Email:** [david.fernxndez.martinez@gmail.com](mailto:david.fernxndez.martinez@gmail.com)  
**LinkedIn:** [linkedin.com/in/david-fernández-martínez](https://linkedin.com/in/david-fernández-martínez)  
**GitHub:** [github.com/davidfernxndez](https://github.com/davidfernxndez)

---

## 03 · Chunking

This notebook demonstrates the **chunking stage** of the RAG pipeline.

Chunking is a critical step in a RAG system because the way documents are divided into smaller pieces directly affects both **retrieval quality and the context provided to the language model**. Chunks that are too large may contain excessive or unrelated information, while chunks that are too small can lose the context required to understand the retrieved content. The goal is therefore to create coherent and sufficiently self-contained chunks that preserve the semantic structure of the original documents.

This project works with GitHub repositories containing different types of content, including source code, Markdown documentation, plain text, and structured configuration files. Since these formats have different structural characteristics, a single chunking strategy is not suitable for all of them. The pipeline therefore applies **file-type-specific chunking strategies** designed to preserve the structure and context of each type of content.

The implementation of the chunking stage is located in `src/github_rag/chunking/`:

* `chunkConfig.py`: Defines the chunking configuration, including chunk size and overlap.
* `chunker.py`: Coordinates the chunking process and applies the appropriate chunker according to the document type.
* `code.py`: Implements the code chunker, which splits source code using language-specific separators provided by LangChain, helping to preserve the structural boundaries of the code.
* `markdown.py`: Implements the Markdown chunker, which first splits documents by Markdown headers and then applies recursive character-based chunking when further splitting is required.
* `text.py`: Implements the text chunker for plain-text and structured files (`.json`, `.toml`, `.yml`, etc.) using a recursive character-based splitting strategy.

**Notebook objectives**

* Inspect how each of the developed chunking strategies works.
* Compare the documents before and after the chunking process.
* Examine how different file types are split according to their structure.
* Inspect the metadata preserved in the resulting chunks to maintain traceability to the original source document.

## Imports and Configuration

In [40]:
from langchain_core.documents import Document
from github_rag.ingestion.repository import GitHubRepositoryLoader
from github_rag.processing.processor import Processor
from github_rag.chunking.chunker import Chunker
from github_rag.chunking.code import CodeChunker
from github_rag.chunking.markdown import MarkdownChunker
from github_rag.chunking.text import TextChunker

You can explore any GitHub repository simply by providing its URL. For this demo, I'll be using one of my own end-to-end Data Science projects, which combines XGBoost and SHAP to build a predictive and explainable system for classifying urban typologies.

In [5]:
REPOSITORY_URL = "https://github.com/davidfernxndez/urban-typology-xai"

## 1. Ingestion and processing

This section apply the Ingestion and processing phase to get the prepared Documents for chunking phase.

In [6]:
# Set the Loader Object with the URL
loader = GitHubRepositoryLoader(REPOSITORY_URL)

# Load the repository files as a list of LangChain Documents
documents = loader.load()

print(f"Number of documents ingested: {len(documents)}")

Number of documents ingested: 28


In [ ]:
# Set the Processor Object
processor = Processor()

# Apply processing to the ingested documents
documents = processor.process(documents)

print(f"Number of documents before processing: {len(documents)}")

Number of documents before processing: 324


## 2. Chunking behaviour

This section demonstrates the different chunking strategies implemented for **source code, Markdown, and text-based files**.

The current configuration uses a chunk size of 1000 characters with a 200-character overlap. These values provide a reasonable balance between preserving contextual information and maintaining sufficiently granular chunks, while remaining configurable for future experimentation.

### 2.1 CodeChunker

The `CodeChunker` uses the **language-specific separators provided by LangChain** to split source code while preserving its natural structure as much as possible. The appropriate separators are selected based on the programming language of the document.

We first apply the `CodeChunker` to a Python source file from this repository and inspect how the original document is divided into smaller chunks.

In [17]:
# Select src/performance_utils.py file that contais 6 methods and 1605 lines of code

code_doc = [doc for doc in documents if doc.metadata['file_path'] == 'src/performance_utils.py'][0]

# Show the Metadata
print("\nMetadata:")
print(code_doc.metadata)
print("\n")

# Show the page content
print("Page content:")
print("-"*30)
print(code_doc.page_content)
print("-"*30)


Metadata:
{'repository': 'davidfernxndez/urban-typology-xai', 'file_path': 'src/performance_utils.py', 'file_type': '.py', 'source': 'https://github.com/davidfernxndez/urban-typology-xai/blob/main/src/performance_utils.py', 'branch': 'main'}


Page content:
------------------------------
"""
Performance evaluation and analysis utilities.

This module implements a Nested Cross-Validation framework for the
robust evaluation of machine learning pipelines. Additionally, it
provides functions for aggregating, analyzing, and visualizing
performance metrics, enabling a comprehensive comparison of model
results.
"""

# ==============================================================================
# IMPORTS
# ==============================================================================

import os
import time
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display
import mlflow

from sklearn.model_selection import GridSea

In [19]:
# Apply the CodeChunker
codechunker = CodeChunker()

code_doc_chunking = codechunker.chunk(code_doc)

print(f"Number of chunks generated: {len(code_doc_chunking)}")

Number of chunks generated: 76


The Python file contains **1,605 lines of code organized into 6 methods**. Using a chunk size of **1,000 characters** with an overlap of **200 characters**, the `CodeChunker` generates **76 chunks**.

We can now inspect the generated chunks to see how the source code has been divided and how the language-specific separators influence the resulting boundaries.

In [21]:
for i, chunk in enumerate(code_doc_chunking[:20]):
    print(f"--- Chunk {i} ---")
    print(f"\n{chunk.page_content}")
    print("\n")

--- Chunk 0 ---

"""
Performance evaluation and analysis utilities.

This module implements a Nested Cross-Validation framework for the
robust evaluation of machine learning pipelines. Additionally, it
provides functions for aggregating, analyzing, and visualizing
performance metrics, enabling a comprehensive comparison of model
results.
"""

# ==============================================================================
# IMPORTS
# ==============================================================================

import os
import time
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display
import mlflow

from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix,
    classification_report
)

from src.config import cfg
# Path to save images in PDF format
OUTPUT_DIR = cfg.OU

We can see that the chunks avoid splitting the code in the middle of a logical structure whenever possible, preserving the organization and context of the source code.

### 2.2 MarkdownChunker

The `MarkdownChunker` uses a dual strategy: it first splits the document by Markdown headers and then applies recursive character splitting when a chunk exceeds the configured chunk size. This approach preserves the Markdown structure for the RAG system, both in the text and in the document metadata. The header hierarchy is stored as additional metadata fields: **Header_1 (#), Header_2 (##), and Header_3 (###)**.

To demonstrate the behaviour of the `MarkdownChunker`, we will explore the `README.md` file, which is relatively long and contains several levels of Markdown headers.

In [26]:
# Select README.md file that contais 259 lines of code

md_doc = [doc for doc in documents if doc.metadata['file_path'] == 'README.md'][0]

# Show the Metadata
print("\nMetadata:")
print(md_doc.metadata)
print("\n")

# Show the page content
print("Page content:")
print("-"*30)
print(md_doc.page_content)
print("-"*30)


Metadata:
{'repository': 'davidfernxndez/urban-typology-xai', 'file_path': 'README.md', 'file_type': '.md', 'source': 'https://github.com/davidfernxndez/urban-typology-xai/blob/main/README.md', 'branch': 'main'}


Page content:
------------------------------
# Explainable Artificial Intelligence for the Study of Urban Fragmentation

### A Comparative Analysis of the Performance and Interpretability of Supervised Learning Models for the Classification of Residential Enclosure Patterns

<p align="left">
  <img src="https://img.shields.io/badge/Language-Python-3776AB?logo=python&logoColor=white" alt="Language: Python">
  <img src="https://img.shields.io/badge/Model-XGBoost-EC6C00" alt="Model: XGBoost">
  <img src="https://img.shields.io/badge/Explainability-SHAP-purple" alt="Explainability: SHAP">
  <img src="https://img.shields.io/badge/Application-Streamlit-FF4B4B?logo=streamlit&logoColor=white" alt="Application: Streamlit">
</p>

<p align="left">
  <img src="images/app_demo/demo.gif" 

In [28]:
# Apply the MarkdownChunker
mdchunker = MarkdownChunker()

md_doc_chunking = mdchunker.chunk(md_doc)

print(f"Number of chunks generated: {len(md_doc_chunking)}")

Number of chunks generated: 25


In [33]:
for i, chunk in enumerate(md_doc_chunking):
    print(f"--- Chunk {i} ---")
    if "header_1" in chunk.metadata:
        print(f"Metadata Header 1: {chunk.metadata['header_1']}")

    if "header_2" in chunk.metadata:
        print(f"Metadata Header 2: {chunk.metadata['header_2']}")

    if "header_3" in chunk.metadata:
        print(f"Metadata Header 3: {chunk.metadata['header_3']}")
    print(f"\n{chunk.page_content}")
    print("\n")

--- Chunk 0 ---
Metadata Header 1: Explainable Artificial Intelligence for the Study of Urban Fragmentation
Metadata Header 3: A Comparative Analysis of the Performance and Interpretability of Supervised Learning Models for the Classification of Residential Enclosure Patterns

# Explainable Artificial Intelligence for the Study of Urban Fragmentation  
### A Comparative Analysis of the Performance and Interpretability of Supervised Learning Models for the Classification of Residential Enclosure Patterns  
<p align="left">
<img src="https://img.shields.io/badge/Language-Python-3776AB?logo=python&logoColor=white" alt="Language: Python">
<img src="https://img.shields.io/badge/Model-XGBoost-EC6C00" alt="Model: XGBoost">
<img src="https://img.shields.io/badge/Explainability-SHAP-purple" alt="Explainability: SHAP">
<img src="https://img.shields.io/badge/Application-Streamlit-FF4B4B?logo=streamlit&logoColor=white" alt="Application: Streamlit">
</p>  
<p align="left">
<img src="images/app_demo

We can see how the chunks capture information structured into sections, which provides greater context to the RAG and enables it to cite the source from which the information was extracted.

### 2.3 TextChunker

`TextChunker` is the simplest chunking strategy. It uses LangChain's recursive character text splitter and is applied to `.txt` files and structured data formats such as `.json`, `.yml`, or `.toml`.

In this repository, these types of files are relatively small, so to demonstrate the behaviour of the `TextChunker`, we manually emulate a larger file.

In [35]:
text_doc = Document(
    page_content="""
This project implements a data processing pipeline for analyzing
large collections of urban data. The pipeline starts by loading
the input data from several sources and validating the structure
of each record.

After validation, the data is cleaned and transformed into a
standardized representation. Missing values are handled according
to predefined rules, categorical variables are encoded, and
numerical variables are normalized when required.

The processed data is then used to generate descriptive statistics
and train several machine learning models. The results are stored
in a structured output directory so they can be used by subsequent
stages of the application.

The system also provides logging functionality to keep track of
the execution of each processing stage. Errors are captured and
reported without stopping the processing of independent records.

Finally, the generated results can be consumed by other components
of the application through a set of predefined interfaces.
""" * 5,
    metadata={
        "repository": "example-repository",
        "file_path": "docs/description.txt",
        "file_type": ".txt",
        "source": "https://github.com/example/example-repository/blob/main/docs/description.txt",
        "branch": "main",
    },
)

# Apply the TextChunker
textchunker = MarkdownChunker()

text_doc_chunking = textchunker.chunk(text_doc)

print(f"Number of chunks generated: {len(text_doc_chunking)}")

for i, chunk in enumerate(text_doc_chunking):
    print(f"--- Chunk {i} ---")
    print(f"\n{chunk.page_content}")
    print("\n")

Number of chunks generated: 6
--- Chunk 0 ---

This project implements a data processing pipeline for analyzing
large collections of urban data. The pipeline starts by loading
the input data from several sources and validating the structure
of each record.

After validation, the data is cleaned and transformed into a
standardized representation. Missing values are handled according
to predefined rules, categorical variables are encoded, and
numerical variables are normalized when required.

The processed data is then used to generate descriptive statistics
and train several machine learning models. The results are stored
in a structured output directory so they can be used by subsequent
stages of the application.

The system also provides logging functionality to keep track of
the execution of each processing stage. Errors are captured and
reported without stopping the processing of independent records.

Finally, the generated results can be consumed by other components
of the applicat

In [37]:
json_doc = Document(
    page_content="""
{
    "project": {
        "name": "Urban Data Analysis",
        "version": "1.0.0",
        "description": "Machine learning pipeline for urban data analysis"
    },
    "configuration": {
        "input_directory": "data/input",
        "output_directory": "data/output",
        "random_seed": 42,
        "test_size": 0.2
    },
    "preprocessing": {
        "missing_values": "median",
        "categorical_encoding": "one_hot",
        "normalization": true
    },
    "models": [
        {
            "name": "logistic_regression",
            "parameters": {
                "max_iter": 1000,
                "class_weight": "balanced"
            }
        },
        {
            "name": "random_forest",
            "parameters": {
                "n_estimators": 200,
                "max_depth": 10,
                "class_weight": "balanced"
            }
        },
        {
            "name": "xgboost",
            "parameters": {
                "n_estimators": 300,
                "max_depth": 6,
                "learning_rate": 0.05
            }
        }
    ],
    "evaluation": {
        "metrics": [
            "accuracy",
            "f1_macro",
            "mcc"
        ],
        "cross_validation": {
            "folds": 5,
            "shuffle": true
        }
    }
}
""" * 3,
    metadata={
        "repository": "example-repository",
        "file_path": "config/model_config.json",
        "file_type": ".json",
        "source": "https://github.com/example/example-repository/blob/main/config/model_config.json",
        "branch": "main",
    },
)

json_doc_chunking = textchunker.chunk(json_doc)

print(f"Number of chunks generated: {len(json_doc_chunking)}")

for i, chunk in enumerate(json_doc_chunking):
    print(f"--- Chunk {i} ---")
    print(f"\n{chunk.page_content}")
    print("\n")

Number of chunks generated: 6
--- Chunk 0 ---

{
    "project": {
        "name": "Urban Data Analysis",
        "version": "1.0.0",
        "description": "Machine learning pipeline for urban data analysis"
    },
    "configuration": {
        "input_directory": "data/input",
        "output_directory": "data/output",
        "random_seed": 42,
        "test_size": 0.2
    },
    "preprocessing": {
        "missing_values": "median",
        "categorical_encoding": "one_hot",
        "normalization": true
    },
    "models": [
        {
            "name": "logistic_regression",
            "parameters": {
                "max_iter": 1000,
                "class_weight": "balanced"
            }
        },
        {
            "name": "random_forest",
            "parameters": {
                "n_estimators": 200,
                "max_depth": 10,
                "class_weight": "balanced"
            }
        },
        {
            "name": "xgboost",
            "parameters": {

## 3. Apply Chunking to the whole repository

In [42]:
print(f"Number of documents before processing: {len(documents)}")

# Set the Chunker object that managed all the specific chunkers
chunker = Chunker()

documents_chunking = chunker.chunk(documents)

print(f"Number of documents before chunking: {len(documents_chunking)}")

Number of documents before processing: 324
Number of documents before chunking: 985
